# Week 57, Day 6: Exam debrief

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w057d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


In [ ]:
import heapq
from collections import Counter

def top_k_frequent(nums, k):
    counts = Counter(nums)
    return [v for _, v in heapq.nsmallest(k, ((-n, v) for v, n in counts.items()))]

print(top_k_frequent([1, 1, 1, 2, 2, 3], 2), top_k_frequent([4, 4, 5, 5, 6], 2))

In [ ]:
def fewest_insertions(s):
    waiting, unmatched = 0, 0
    for ch in s:
        if ch == "(":
            waiting += 1
        elif waiting:
            waiting -= 1
        else:
            unmatched += 1
    return waiting + unmatched

print(fewest_insertions("())"), fewest_insertions("()))(("))

In [ ]:
a = [0, 1, 2, 4, 8]
b = [1, 3, 4, 5, 8]
c = 6
d = 1
e = 4

In [ ]:
import heapq

def alphabet_order(words):
    letters = {ch for w in words for ch in w}
    after = {ch: set() for ch in letters}
    for a, b in zip(words, words[1:]):
        for x, y in zip(a, b):
            if x != y:
                after[x].add(y)
                break
        else:
            if len(a) > len(b):
                return None
    need = {ch: 0 for ch in letters}
    for x in after:
        for y in after[x]:
            need[y] += 1
    ready = [ch for ch in letters if need[ch] == 0]
    heapq.heapify(ready)
    out = []
    while ready:
        ch = heapq.heappop(ready)
        out.append(ch)
        for y in after[ch]:
            need[y] -= 1
            if need[y] == 0:
                heapq.heappush(ready, y)
    return "".join(out) if len(out) == len(letters) else None

print(alphabet_order(["wrt", "wrf", "er", "ett", "rftt"]), alphabet_order(["abc", "ab"]))

In [ ]:
import heapq
from collections import Counter

def least_slots(tasks, n):
    if not tasks:
        return 0
    c = Counter(tasks)
    most = max(c.values())
    ties = sum(1 for v in c.values() if v == most)
    return max(len(tasks), (most - 1) * (n + 1) + ties)

def schedule(tasks, n):
    heap = [(-count, ch) for ch, count in Counter(tasks).items()]
    heapq.heapify(heap)
    cooling = []                       # (time it's free again, -count, ch)
    out, time = [], 0
    while heap or cooling:
        while cooling and cooling[0][0] <= time:
            _, count, ch = heapq.heappop(cooling)
            heapq.heappush(heap, (count, ch))
        if heap:
            count, ch = heapq.heappop(heap)
            out.append(ch)
            if count + 1 < 0:
                heapq.heappush(cooling, (time + n + 1, count + 1, ch))
        else:
            out.append(".")
        time += 1
    return "".join(out)

print(least_slots("AAABBB", 2), schedule("AAABBB", 2), schedule("AAAB", 1))

In [ ]:
def longest_balanced(s):
    stack, best = [-1], 0
    for i, ch in enumerate(s):
        if ch == "(":
            stack.append(i)
        else:
            stack.pop()
            if not stack:
                stack.append(i)          # a new base after an unmatched ")"
            else:
                best = max(best, i - stack[-1])
    return best

print(longest_balanced(")()())"), longest_balanced("()(())"))

In [ ]:
def first_at_least(nums, target):
    lo, hi = 0, len(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] >= target:
            hi = mid
        else:
            lo = mid + 1
    return lo

def count_between(nums, lo, hi):
    return first_at_least(nums, hi + 1) - first_at_least(nums, lo)

nums = [1, 3, 3, 5, 8, 9]
print([first_at_least(nums, 4), first_at_least(nums, 8), count_between(nums, 3, 5)])

In [ ]:
def count_xor_stretches(nums, k):
    seen, prefix, total = {0: 1}, 0, 0
    for x in nums:
        prefix ^= x
        total += seen.get(prefix ^ k, 0)
        seen[prefix] = seen.get(prefix, 0) + 1
    return total

def longest_xor_stretch(nums, k):
    first, prefix, best = {0: -1}, 0, 0
    for i, x in enumerate(nums):
        prefix ^= x
        if prefix ^ k in first:
            best = max(best, i - first[prefix ^ k])
        first.setdefault(prefix, i)
    return best

print(count_xor_stretches([4, 2, 2, 6, 4], 6), longest_xor_stretch([4, 2, 2, 6, 4], 6))

In [ ]:
import heapq

class OrderBook:
    def __init__(self):
        self.bids = []           # (-price, id)
        self.asks = []           # (price, id)
        self.orders = {}         # id -> [side, trader, price, quantity]
        self.next_id = 1
        self._trades = []

    def _check(self, price, quantity):
        for v in (price, quantity):
            if not isinstance(v, int) or isinstance(v, bool) or v <= 0:
                raise ValueError("prices and quantities must be positive whole numbers")

    def _top(self, heap):
        while heap and heap[0][1] not in self.orders:
            heapq.heappop(heap)            # skip cancelled or filled orders
        return heap[0] if heap else None

    def _place(self, side, trader, price, quantity):
        self._check(price, quantity)
        oid = self.next_id
        self.next_id += 1
        other = self.asks if side == "buy" else self.bids
        while quantity:
            top = self._top(other)
            if top is None:
                break
            resting_price = top[0] if side == "buy" else -top[0]
            if (side == "buy" and resting_price > price) or (side == "sell" and resting_price < price):
                break
            rest = self.orders[top[1]]
            q = min(quantity, rest[3])
            if side == "buy":
                self._trades.append((trader, rest[1], resting_price, q))
            else:
                self._trades.append((rest[1], trader, resting_price, q))
            quantity -= q
            rest[3] -= q
            if rest[3] == 0:
                del self.orders[top[1]]
                heapq.heappop(other)
        if quantity:
            self.orders[oid] = [side, trader, price, quantity]
            if side == "buy":
                heapq.heappush(self.bids, (-price, oid))
            else:
                heapq.heappush(self.asks, (price, oid))
        return oid

    def buy(self, trader, price, quantity):
        return self._place("buy", trader, price, quantity)

    def sell(self, trader, price, quantity):
        return self._place("sell", trader, price, quantity)

    def trades(self):
        return list(self._trades)

    def cancel(self, order_id):
        if order_id not in self.orders:
            raise KeyError(order_id)
        del self.orders[order_id]

    def best_bid(self):
        top = self._top(self.bids)
        return -top[0] if top else None

    def best_ask(self):
        top = self._top(self.asks)
        return top[0] if top else None

    def depth(self, side, levels):
        totals = {}
        for side_, _, price, qty in self.orders.values():
            if side_ == side:
                totals[price] = totals.get(price, 0) + qty
        return sorted(totals.items(), reverse=(side == "buy"))[:levels]

book = OrderBook()
book.sell("s1", 101, 2)
book.sell("s2", 100, 2)
book.buy("b1", 101, 3)
print(book.trades(), book.best_ask(), book.depth("sell", 2))

### Exercise: Second chance: the least common

Write `rarest_words(text, k)`: the k words (lower-cased runs of letters) that appear **least** often, rarest first, ties alphabetical.

In [ ]:
import re
from collections import Counter

def rarest_words(text, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from collections import Counter\nimport random\ndef test_rare():\n    \"counting again\"\n    assert rarest_words(\"the cat and the hat and the bat\", 2) == [\"bat\", \"cat\"] and rarest_words(\"\", 3) == []\n    assert rarest_words(\"A a b\", 5) == [\"b\", \"a\"]\n    rng = random.Random(1)\n    for _ in range(100):\n        words = [rng.choice([\"ab\", \"cd\", \"ef\", \"gh\", \"ij\"]) for _ in range(rng.randint(0, 15))]\n        k = rng.randint(0, 4)\n        c = Counter(words)\n        assert rarest_words(\" \".join(words), k) == sorted(c, key=lambda w: (c[w], w))[:k]")

### Exercise: Second chance: an order from rankings

Several judges each ranked some of the contestants (a list, best first). Write `combined_order(rankings)`: one order of every contestant consistent with **all** the rankings, choosing the alphabetically first available contestant each time, or `None` if the rankings contradict each other.

In [ ]:
import heapq

def combined_order(rankings):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef test_order():\n    \"topological order again\"\n    assert combined_order([[\"ann\", \"bo\"], [\"bo\", \"cy\"]]) == [\"ann\", \"bo\", \"cy\"]\n    assert combined_order([[\"a\", \"b\"], [\"b\", \"a\"]]) is None and combined_order([]) == [] and combined_order([[\"x\"]]) == [\"x\"]\n    rng = random.Random(2)\n    for _ in range(60):\n        names = rng.sample(\"abcdef\", rng.randint(1, 5))\n        rankings = [rng.sample(names, rng.randint(1, len(names))) for _ in range(rng.randint(1, 3))]\n        everyone = sorted({n for r in rankings for n in r})\n        ok = [list(p) for p in itertools.permutations(everyone)\n              if all(p.index(a) < p.index(b) for r in rankings for a, b in zip(r, r[1:]))]\n        assert combined_order(rankings) == (min(ok) if ok else None)")

### Exercise: Second chance: removing to balance

Write `remove_to_balance(s)`: a string of brackets and letters with the **fewest** brackets removed so that the brackets balance, keeping the leftmost possible brackets (letters always stay). Use a stack of the indices of unmatched `(`.

In [ ]:
def remove_to_balance(s):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef balanced(t):\n    d = 0\n    for ch in t:\n        if ch == \"(\":\n            d += 1\n        elif ch == \")\":\n            d -= 1\n            if d < 0:\n                return False\n    return d == 0\ndef test_remove():\n    \"a stack again\"\n    assert remove_to_balance(\"a)b(c)d\") == \"ab(c)d\" and remove_to_balance(\"))((\") == \"\" and remove_to_balance(\"(a(b)\") == \"a(b)\"\n    rng = random.Random(3)\n    for _ in range(150):\n        s = \"\".join(rng.choice(\"()ab\") for _ in range(rng.randint(0, 10)))\n        got = remove_to_balance(s)\n        brackets = sum(1 for ch in s if ch in \"()\")\n        best = 0\n        for k in range(brackets, -1, -1):\n            found = False\n            idx = [i for i, ch in enumerate(s) if ch in \"()\"]\n            for keep in itertools.combinations(idx, k):\n                t = \"\".join(ch for i, ch in enumerate(s) if ch not in \"()\" or i in keep)\n                if balanced(t):\n                    found = True\n                    break\n            if found:\n                best = k\n                break\n        assert balanced(got) and sum(1 for ch in got if ch in \"()\") == best and got.replace(\"(\", \"\").replace(\")\", \"\") == s.replace(\"(\", \"\").replace(\")\", \"\")")

### Exercise: Second chance: the last at most

Write `last_at_most(nums, target)`: in a sorted list, the index of the **last** value at most target, or −1, with a half-open binary search (no `bisect`).

In [ ]:
def last_at_most(nums, target):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import bisect, random\ndef test_last():\n    \"binary search again\"\n    assert last_at_most([1, 3, 3, 5], 3) == 2 and last_at_most([1, 3], 0) == -1 and last_at_most([], 5) == -1 and last_at_most([2, 4], 9) == 1\n    rng = random.Random(4)\n    for _ in range(300):\n        a = sorted(rng.randint(0, 9) for _ in range(rng.randint(0, 9)))\n        t = rng.randint(-1, 10)\n        assert last_at_most(a, t) == bisect.bisect_right(a, t) - 1\n    assert \"bisect\" not in source")

### Exercise: Second chance: even stretches

Write `count_even_sum_stretches(nums)`: how many non-empty stretches of neighbouring whole numbers have an **even** sum, in O(n): only each prefix sum's parity matters, so count the prefixes of each parity.

In [ ]:
def count_even_sum_stretches(nums):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_even():\n    \"prefix parities\"\n    assert count_even_sum_stretches([1, 2, 3]) == 2 and count_even_sum_stretches([]) == 0 and count_even_sum_stretches([2, 4]) == 3\n    rng = random.Random(5)\n    for _ in range(200):\n        nums = [rng.randint(-5, 5) for _ in range(rng.randint(0, 12))]\n        assert count_even_sum_stretches(nums) == sum(1 for i in range(len(nums)) for j in range(i + 1, len(nums) + 1) if sum(nums[i:j]) % 2 == 0)")

### Exercise: Second chance: a waiting list

A clinic keeps a waiting list where patients have a priority (lower is more urgent) and arrive in order. Write `WaitingList()` with `add(name, priority)`, `next()` (remove and return the most urgent, earliest arrival first; `None` if empty), `remove(name)` (`KeyError` if not waiting), and `len()`. Use a heap with lazy deletion, as in the order book. Names are unique while waiting, but a name may come back after leaving.

In [ ]:
import heapq

class WaitingList:
    def __init__(self):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_waiting():\n    \"heap and lazy deletion\"\n    w = WaitingList()\n    w.add(\"ann\", 2)\n    w.add(\"bo\", 1)\n    w.add(\"cy\", 2)\n    w.remove(\"bo\")\n    assert len(w) == 2 and w.next() == \"ann\" and w.next() == \"cy\" and w.next() is None\n    w.add(\"bo\", 5)\n    assert w.next() == \"bo\"\n    try:\n        w.remove(\"zed\")\n        assert False\n    except KeyError:\n        pass\n    rng = random.Random(6)\n    for _ in range(30):\n        w, model, t = WaitingList(), [], 0\n        for step in range(50):\n            r = rng.random()\n            if r < 0.5:\n                name = f\"p{step}\"\n                pr = rng.randint(1, 3)\n                w.add(name, pr)\n                model.append((pr, t, name))\n                t += 1\n            elif r < 0.7 and model:\n                victim = rng.choice(model)\n                w.remove(victim[2])\n                model.remove(victim)\n            else:\n                want = min(model)[2] if model else None\n                if model:\n                    model.remove(min(model))\n                assert w.next() == want\n            assert len(w) == len(model)")

## Your score

Write down your total before reading on. Then, as you go through each problem, note **why** you lost any marks:

1. **Didn't know it**: a gap in what you've learned. Revise the topic.
2. **Picked the wrong pattern**: you had the tools but reached for the wrong one. Note what in the problem should have pointed you to the right one.
3. **Edge case**: the main case worked, but empty input, a tie or a boundary failed.
4. **Ran out of time**: the plan, not the knowledge, was the problem.

| Score | What it means |
|---|---|
| 144 or more (80%) | Distinction. You're well prepared for Phase 5. |
| 108 to 143 (60%) | Pass. Revise the topics where you lost marks during the rest week. |
| under 108 | Not yet. Spend the rest week on the topics below, then take the exam again. What matters is the second attempt. |

## Part A

| Questions | The idea | Revise |
|---|---|---|
| 1 | amortised appends | [Week 41, Day 2](w041d2.html) |
| 2 | sets for "seen before" | [Week 44, Day 1](w044d1.html) |
| 3 | fast and slow pointers | [Week 42, Day 5](w042d5.html) |
| 4 | bracket matching with a stack | [Week 43, Day 1](w043d1.html) |
| 5 | monotonic stacks | [Week 43, Day 3](w043d3.html) |
| 6, 7 | BST height and traversal orders | [Week 45, Day 2](w045d2.html), [Week 46, Day 1](w046d1.html) |
| 8 | heaps pop smallest first | [Week 47, Day 1](w047d1.html) |
| 9 | tries cost O(L) | [Week 48, Day 1](w048d1.html) |
| 10 | BFS for fewest edges | [Week 50, Day 2](w050d2.html) |
| 11 | why Dijkstra needs non-negative weights | [Week 51, Day 1](w051d1.html) |
| 12, 13 | union-find and spanning trees | [Week 51, Days 3 and 4](w051d3.html) |
| 14 | earliest finish first | [Week 52, Day 2](w052d2.html) |
| 15 | choose, explore, unchoose | [Week 52, Day 3](w052d3.html) |
| 16 | counting coin collections, coins outside | [Week 53, Day 2](w053d2.html) |
| 17 | the LCS table | [Week 54, Day 1](w054d1.html) |
| 18 | Held-Karp's states | [Week 54, Day 4](w054d4.html) |
| 19 | x & (x − 1) and x & −x | [Week 55, Day 2](w055d2.html) |
| 20 | modular inverses | [Week 55, Day 3](w055d3.html) |

Question 16 catches people who count orders instead of collections: with coins on the **outside** loop, 1 + 2 + 2 and 2 + 1 + 2 are the same collection.

## Part B

Each model solution runs, so try it. The notes give the key decision and the most common mistakes.

### B1: The most common

**The decision**: count with a Counter, then take the k best by (−count, value) with `heapq.nsmallest`, which keeps a heap of size k: O(n log k). **Common mistakes**: sorting by count alone, so ties come out in the wrong order; `Counter.most_common`, which breaks ties by first appearance, not by value.

### B2: Mending brackets

**The decision**: one pass with two counters: opens still waiting for a close, and closes that arrived with nothing to close. Both need an insertion. **Common mistakes**: returning `abs(opens - closes)`, which says `")("` needs 0.

### B3: Predict

**The answers**: a: 0 passes `x & (x − 1) == 0` too, so `[0, 1, 2, 4, 8]`; b: in-order of a BST is sorted, `[1, 3, 4, 5, 8]`; c: a spanning tree of n nodes has n − 1 edges, 6; d: one replacement, 1; e: 1, 2, 3, 8 (or 5, 6, 7, 8), 4.

### B4: An unknown alphabet

**The decision**: compare each neighbouring pair of words; the first differing letters give one "comes before" edge. Then Kahn's algorithm with a heap for the smallest available letter (Weeks 50 and 47). **Common mistakes**: comparing every pair of words (only neighbours matter); missing the prefix case ("abc" before "ab" is impossible); forgetting letters with no edges.

### B5: Tasks with a cool-down

**The decision**: the count is a formula: with the most frequent letter appearing `most` times (and `ties` letters sharing that count), the frame `(most − 1) × (n + 1) + ties` is a lower bound, and when there are more tasks than that, no idle slot is needed. The schedule is greedy with two heaps: ready letters by remaining count, and cooling letters by the time they're free. **Common mistakes**: running letters in plain alphabetical order, which wastes slots.

### B6: The longest balanced stretch

**The decision**: a stack of indices of unmatched brackets, with the index just before the current stretch at the bottom (−1 to start). After a `)` pops its partner, the stretch ends at i and starts after the new top. **Common mistakes**: counting matched pairs overall, which joins stretches separated by an unmatched `)`.

### B7: Three bugs

**The bugs**: with a half-open search `[lo, hi)` the update on success must be `hi = mid` (not `mid − 1`), and the loop `while lo < hi` (the extra guard hid the problem); and values equal to `hi` are counted by searching for `hi + 1`.

### B8: Stretches with a given XOR

**The decision**: prefix XORs with a dict, exactly like prefix sums for "sum equals k" (Week 41), because `prefix[j] ^ prefix[i]` is the XOR of `nums[i:j]`, and `a ^ b == k` exactly when `a ^ k == b`. Count with a dict of counts; for the longest, keep each prefix's **first** position. **Common mistakes**: an O(n²) double loop.

## Part C

**The decisions**: two heaps keyed by (price, id) for asks and (−price, id) for bids, so "best price, then oldest" is the heap order; a dict of resting orders by id; trades at the resting price; cancelling just deletes from the dict, and `_top` throws away heap entries whose ids are gone (lazy deletion, Week 47). **Common mistakes**: trading at the new order's price; forgetting to pop a fully filled order from its heap; removing a cancelled order from the middle of a heap (O(n) and easy to get wrong).

## Second chances

A fresh problem for each skill the exam tested. If you lost marks on a problem above, do its partner here without looking back.

## What's next

**Your revision plan.** For each problem where you lost marks, write one line: the problem, the kind of mistake, and the day you'll revisit. Do those days during the rest week. If you're retaking the exam, do it after the rest week, without looking at this page again first.

**Phase 5: Advanced Python.** After a week's rest, Phase 5 turns back to the language itself, with sixteen weeks of algorithms behind you. It opens with the **iteration protocol**: what a `for` loop really does, iterators and generators (the `yield` you've been promised since Phase 2), generator pipelines and `itertools`. Then functions in depth: closures, `nonlocal`, decorators and `functools`. You'll find that many of this phase's iterator classes and "one-item lists" become one line each.

You've finished Phase 4. Sixteen weeks ago you asked why appending to a list is fast; since then you've built arrays, linked lists, stacks, queues, hash tables, trees, heaps and tries, searched graphs every way that matters, and learned to choose between greed, search and dynamic programming. Take the rest week: you've earned it.